# 04 - Building Custom Simulator Adapters

In this advanced guide, you will learn how to wrap third-party software engines, REST APIs, or compiled C/C++ libraries into an `energysim` co-simulation adapter by subclassing `SimulatorAdapter`:

1. The `SimulatorAdapter` contract: `init()`, `set_value()`, `step()`, `get_value()`.
2. Declaring the adapter's inputs and outputs and connecting it to other simulators in a `world`.
3. Wrapping an external mock thermodynamic fluid flow solver.


In [ ]:
from typing import List, Any, Dict, Optional
from energysim import world
from energysim.base import SimulatorAdapter

print("Base classes imported!")


## Step 1: Implement the Custom Adapter


In [ ]:
class MockHydraulicNetworkAdapter(SimulatorAdapter):
    """
    Adapter wrapping an external hydraulic pipe network solver.
    Simulates pipe flow, pressure drop, and fluid temperature.
    """
    def __init__(self, sim_name: str, step_size: float = 60.0):
        super().__init__()
        self.sim_name = sim_name
        self.step_size = step_size
        self.mass_flow_kg_s = 10.0
        self.temp_in_c = 80.0
        self.pressure_drop_bar = 1.2
        self.temp_out_c = 75.0

    def init(self) -> None:
        pass

    def set_value(self, parameters: List[str], values: Any) -> None:
        if not isinstance(values, (list, tuple)):
            values = [values] * len(parameters)
        for param, val in zip(parameters, values):
            if param == "m_dot_in":
                self.mass_flow_kg_s = float(val)
            elif param == "temp_in":
                self.temp_in_c = float(val)

    def step(self, time: float, step_size: Optional[float] = None) -> None:
        # Hydraulic friction: delta_P ~ m_dot^2
        self.pressure_drop_bar = 0.012 * (self.mass_flow_kg_s ** 2)
        # Thermal loss through insulation
        self.temp_out_c = self.temp_in_c - (0.5 * 30.0 / max(self.mass_flow_kg_s, 0.1))

    def get_value(self, parameters: List[str], time: Optional[float] = None) -> List[Any]:
        res = []
        for p in parameters:
            if p == "pressure_drop_bar":
                res.append(self.pressure_drop_bar)
            elif p == "temp_out":
                res.append(self.temp_out_c)
        return res

print("MockHydraulicNetworkAdapter implemented!")


## Step 2: Co-Simulate the Hydraulic Network in World


In [ ]:
sim = world(start_time=0, stop_time=1800, t_macro=60)

pipe = MockHydraulicNetworkAdapter(sim_name="district_heating_pipe")
sim.add_model("pipe", model=pipe, inputs=["m_dot_in", "temp_in"], outputs=["pressure_drop_bar", "temp_out"])

sim.add_signal("flow_ramp", lambda t: 5.0 + 10.0 * (t / 1800.0))
sim.add_signal("temp_source", lambda t: 85.0)

sim.add_connections({
    "flow_ramp.y": "pipe.m_dot_in",
    "temp_source.y": "pipe.temp_in",
})

sim.simulate()
results = sim.last_results
display(results["pipe"].head())
